In [0]:
import os

def carpeta_data():
    ruta = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
    if not ruta.startswith("/Workspace"):
        ruta = "/Workspace" + ruta
    os.makedirs(f"{ruta}/data", exist_ok=True)
    return f"{ruta}/data"

def guardar_gold(df, nombre):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"workspace.pobreza.{nombre}_gold")
    df.toPandas().to_json(f"{carpeta_data()}/{nombre}.json", orient="records", force_ascii=False, indent=1)
    print("OK:", nombre)

In [0]:
from pyspark.sql import Row

F23 = "MSP – Natalidad y mortalidad infantil 2023, informe preliminar"
F24 = "MSP – Estadísticas vitales 2024, datos preliminares"
datos = [(2015,48926,F23),(2016,47058,F23),(2017,43036,F23),(2018,40139,F23),(2019,37472,F23),
         (2020,35874,F23),(2021,34603,F23),(2022,32301,F23),(2023,31381,F23),(2024,29899,F24)]
nac = spark.createDataFrame([Row(anio=a, nacimientos=n, fuente=f) for a, n, f in datos])

nac.write.mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.pobreza.nacimientos_silver")
guardar_gold(nac.select("anio", "nacimientos").orderBy("anio"), "nacimientos_por_anio")

display(nac)